# Inequality Enumeration Pipeline: D1 → D2

This notebook demonstrates the full pipeline for discovering universal bond-percolation
inequalities from the decision-tree feasibility oracle (D1, `PercolationOracle`) through the
projected cone oracle (D2, `ProjectedConeOracle`).

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..", "PercolationOracle"))
Pkg.develop(path=joinpath(@__DIR__, "..", "ProjectedConeOracle"))
Pkg.instantiate()

using PercolationOracle
using ProjectedConeOracle
using Printf

# Helper: run enumeration with solver output suppressed
function enumerate_quiet(n_obs, m; kwargs...)
    result = redirect_stdout(devnull) do
        redirect_stderr(devnull) do
            enumerate_all_inequalities(n_obs, m; verbose=false, kwargs...)
        end
    end
    return result
end;

## Section 1: Decision Tree Enumeration for $n=3$

For $n=3$ vertices, there are $B_3 = 5$ connectivity partitions:

| ID | Partition | Notation |
|----|-----------|----------|
| 0  | $\{1,2,3\}$ | $123$ |
| 1  | $\{1,2\}\{3\}$ | $12\mid3$ |
| 2  | $\{1,3\}\{2\}$ | $13\mid2$ |
| 3  | $\{1\}\{2,3\}$ | $1\mid23$ |
| 4  | $\{1\}\{2\}\{3\}$ | $1\mid2\mid3$ |

The paper uses $m=4$ decision trees $(T_0, T_1, T_2, T_3)$. We show how $|F|$ grows as we include more trees.

In [2]:
raw_tuples = load_feasible_tuples()
n_obs = 3

println("Partition labels for n=$n_obs:")
for pid in all_partition_ids(n_obs)
    println("  ID $pid \u2192 $(partition_label(n_obs, pid))")
end

println("\nFeasible set sizes |F| for m = 1, 2, 3, 4:")
for m in 1:4
    projected = Set{Any}()
    for t in raw_tuples
        pairs = archive_tuple_to_paper_pairs(t, n_obs, 4)
        push!(projected, Tuple(pairs[i] for i in 1:m))
    end
    println("  m = $m:  |F| = $(length(projected))")
end

Partition labels for n=3:


  ID 0 → 123
  ID 1 → 12|3
  ID 2 → 13|2
  ID 3 → 1|23
  ID 4 → 1|2|3

Feasible set sizes |F| for m = 1, 2, 3, 4:
  m = 1:  |F| = 25


  m = 2:  |F| = 139
  m = 3:  |F| = 570


  m = 4:  |F| = 1265


The full paper family achieves $|F(T_0, T_1, T_2, T_3)| = 1265$ feasible partition tuples,
matching Section 10 of the paper. Each tuple is a sequence of $m$ partition pairs
$((p_1, \bar{p}_1), \ldots, (p_m, \bar{p}_m))$ with $p_k, \bar{p}_k \in \mathcal{J}_3$.

## Section 2: Inequality Enumeration for $m=2$

With only $m=2$ trees, the feasible-potential cone projects to a cone with 15 extreme rays in the
15-dimensional symmetric coordinate space $\binom{B_3+1}{2} = 15$.

In [3]:
result_m2 = enumerate_quiet(3, 2)

println("m=2 results:")
println("  Extreme rays (inequalities): $(length(result_m2.rays))")
println("  Facet normals (meta-constraints): $(length(result_m2.facet_normals))")
println("  Computation time: $(round(result_m2.timing, digits=2))s")
println("\nSymmetric coordinate labels:")
for (i, label) in enumerate(result_m2.labels)
    println("  [$i] $label")
end

m=2 results:
  Extreme rays (inequalities): 15
  Facet normals (meta-constraints): 15
  Computation time: 0.91s

Symmetric coordinate labels:
  [1] mu(123)^2
  [2] mu(12|3)^2
  [3] mu(13|2)^2
  [4] mu(1|23)^2
  [5] mu(1|2|3)^2
  [6] mu(123)*mu(12|3)
  [7] mu(123)*mu(13|2)
  [8] mu(123)*mu(1|23)
  [9] mu(123)*mu(1|2|3)
  [10] mu(12|3)*mu(13|2)
  [11] mu(12|3)*mu(1|23)
  [12] mu(12|3)*mu(1|2|3)
  [13] mu(13|2)*mu(1|23)
  [14] mu(13|2)*mu(1|2|3)
  [15] mu(1|23)*mu(1|2|3)


In [4]:
println("All $(length(result_m2.formatted_inequalities)) extremal inequalities for m=2:\n")
for (i, ineq) in enumerate(result_m2.formatted_inequalities)
    println("  ($i)  $ineq")
end

All 15 extremal inequalities for m=2:

  (1)  mu(123)*mu(1|2|3) - mu(12|3)*mu(13|2) - mu(12|3)*mu(1|23) >= 0
  (2)  mu(1|23)*mu(1|2|3) >= 0
  (3)  mu(13|2)*mu(1|23) >= 0
  (4)  mu(12|3)*mu(1|2|3) >= 0
  (5)  mu(123)*mu(13|2) >= 0
  (6)  mu(123)*mu(12|3) >= 0
  (7)  mu(123)*mu(1|23) >= 0
  (8)  mu(1|2|3)^2 >= 0
  (9)  mu(1|23)^2 >= 0
  (10)  mu(12|3)*mu(13|2) >= 0
  (11)  mu(13|2)^2 >= 0
  (12)  mu(13|2)*mu(1|2|3) >= 0
  (13)  mu(12|3)^2 >= 0
  (14)  mu(12|3)*mu(1|23) >= 0
  (15)  mu(123)^2 >= 0


## Section 3: Full Paper Family ($m=4$, $|F|=1265$)

The main computational result: using all four decision trees from the paper, the pipeline
discovers **17 extremal inequalities**, including Equation (12):

$$-\mu(12|3)\mu(13|2) - \mu(12|3)\mu(1|23) - \mu(13|2)\mu(1|23) + \mu(123)\mu(1|2|3) \geq 0$$

In [5]:
result_m4 = enumerate_quiet(3, 4)

println("m=4 results (paper family):")
println("  Feasible tuples: |F| = 1265")
println("  Extreme rays (inequalities): $(length(result_m4.rays))")
println("  Facet normals (meta-constraints): $(length(result_m4.facet_normals))")
println("  Computation time: $(round(result_m4.timing, digits=2))s")
println("  Oracle iterations: $(result_m4.projection_result.iterations)")
println("  Total LP calls: $(result_m4.projection_result.lp_calls)")

m=4 results (paper family):
  Feasible tuples: |F| = 1265
  Extreme rays (inequalities): 17
  Facet normals (meta-constraints): 19
  Computation time: 3.88s
  Oracle iterations: 2
  Total LP calls: 21


In [6]:
println("All $(length(result_m4.formatted_inequalities)) extremal inequalities for m=4:\n")
for (i, ineq) in enumerate(result_m4.formatted_inequalities)
    println("  ($i)  $ineq")
end

All 17 extremal inequalities for m=4:

  (1)  mu(123)*mu(1|2|3) - mu(12|3)*mu(13|2) - mu(12|3)*mu(1|23) - mu(13|2)*mu(1|23) >= 0
  (2)  mu(13|2)*mu(1|2|3) >= 0
  (3)  mu(12|3)*mu(1|2|3) >= 0
  (4)  mu(1|23)*mu(1|2|3) >= 0
  (5)  mu(123)*mu(1|23) >= 0
  (6)  mu(123)*mu(13|2) >= 0
  (7)  mu(123)*mu(12|3) >= 0
  (8)  mu(13|2)*mu(1|23) >= 0
  (9)  mu(1|2|3)^2 >= 0
  (10)  mu(1|23)^2 >= 0
  (11)  mu(13|2)^2 >= 0
  (12)  mu(12|3)*mu(1|23) >= 0
  (13)  mu(12|3)^2 >= 0
  (14)  mu(12|3)*mu(13|2) >= 0
  (15)  mu(123)*mu(12|3) + mu(123)*mu(13|2) + mu(123)*mu(1|23) - mu(123)*mu(1|2|3) + 2*mu(12|3)*mu(13|2) + mu(12|3)*mu(1|23) + mu(13|2)*mu(1|23) + mu(1|23)*mu(1|2|3) >= 0
  (16)  mu(1|23)^2 + mu(123)*mu(12|3) + mu(123)*mu(13|2) - mu(123)*mu(1|2|3) + 2*mu(12|3)*mu(13|2) + mu(12|3)*mu(1|23) + mu(13|2)*mu(1|23) + mu(1|23)*mu(1|2|3) >= 0
  (17)  mu(123)^2 >= 0


In [7]:
# Identify Equation (12) among the rays
eq12_found = false
for (i, ineq) in enumerate(result_m4.formatted_inequalities)
    sig = canonicalize_integer_ray(result_m4.rays[i]; tol=1e-8, normalize_sign=true)
    nonzero = [(j, v) for (j, v) in enumerate(sig) if v != 0]
    if length(nonzero) == 4
        vals = sort([v for (_, v) in nonzero])
        if vals == [-1, -1, -1, 1]
            global eq12_found = true
            println("Equation (12) found as ray #$i:")
            println("  $ineq")
            println("\nInteger coefficient vector (canonical):")
            println("  $sig")
        end
    end
end
eq12_found || println("WARNING: Equation (12) not found among the rays")

Equation (12) found as ray #1:


  mu(123)*mu(1|2|3) - mu(12|3)*mu(13|2) - mu(12|3)*mu(1|23) - mu(13|2)*mu(1|23) >= 0

Integer coefficient vector (canonical):
  BigInt[0, 0, 0, 0, 0, 0, 0, 0, 1, -1, -1, 0, -1, 0, 0]


true

## Section 4: Interpretation — Rays vs. Facets

The projected cone oracle returns both **extreme rays** and **facet normals** of the
projected coefficient cone $D = L(K_{\text{fp}})$. These are fundamentally different objects:

- **Extreme rays of $D$** = valid universal percolation inequalities (coefficient vectors
  $\Phi$ such that $\sum_{p,q} \Phi(p,q)\,\mu(p)\mu(q) \geq 0$ for all bond percolation measures $\mu$).
- **Facet normals of $D$** = constraints on the coefficient space (meta-constraints describing
  which $\Phi$ vectors are achievable via feasible potentials).

This is the standard primal-dual duality for polyhedral cones: rays of $D$ correspond to
facets of $D^*$, and vice versa. The production code correctly extracts inequalities from `result.rays`.

In [8]:
println("Rays vs. Facets comparison for m=4:")
println("  Extreme rays (= inequalities on \u03bc): $(length(result_m4.rays))")
println("  Facet normals (= meta-constraints):  $(length(result_m4.facet_normals))")
println()

ray_sigs = Set([canonicalize_integer_ray(r; tol=1e-8, normalize_sign=true) for r in result_m4.rays])
facet_sigs = Set([canonicalize_integer_ray(f; tol=1e-8, normalize_sign=true) for f in result_m4.facet_normals])

common = length(intersect(ray_sigs, facet_sigs))
println("  Signatures in common: $common")
println("  Signatures only in rays: $(length(setdiff(ray_sigs, facet_sigs)))")
println("  Signatures only in facets: $(length(setdiff(facet_sigs, ray_sigs)))")
println()
println("The rays and facets are distinct objects \u2014 confirming that using facets")
println("as inequalities would be a primal\u2013dual error.")

Rays vs. Facets comparison for m=4:
  Extreme rays (= inequalities on μ): 17
  Facet normals (= meta-constraints):  19

  Signatures in common: 11


  Signatures only in rays: 6
  Signatures only in facets: 8

The rays and facets are distinct objects — confirming that using facets
as inequalities would be a primal–dual error.


### Coefficient-to-polynomial translation

Each extreme ray $\Phi \in \mathbb{R}^{15}$ is a coefficient vector for a quadratic form
in the connectivity partition law $\mu$. The 15 coordinates correspond to:
- 5 diagonal terms: $\mu(p)^2$ for each partition $p$
- 10 off-diagonal terms: $\mu(p)\mu(q)$ for each pair $p < q$

The inequality is $\sum_{p \leq q} \Phi(p,q) \cdot \mu(p)\mu(q) \geq 0$, valid for all
bond percolation measures on $K_3$.

In [9]:
# Classify m=4 inequalities by structure
println("Classification of m=4 extremal inequalities:\n")
trivial = 0
nontrivial = 0
for (i, ray) in enumerate(result_m4.rays)
    sig = canonicalize_integer_ray(ray; tol=1e-8, normalize_sign=true)
    nonzero_count = count(x -> x != 0, sig)
    if nonzero_count == 1
        trivial += 1
    else
        nontrivial += 1
        println("  Non-trivial #$nontrivial (ray $i): $(result_m4.formatted_inequalities[i])")
    end
end
println("\nSummary: $trivial trivial positivity + $nontrivial non-trivial = $(trivial + nontrivial) total")

Classification of m=4 extremal inequalities:

  Non-trivial #1 (ray 1): mu(123)*mu(1|2|3) - mu(12|3)*mu(13|2) - mu(12|3)*mu(1|23) - mu(13|2)*mu(1|23) >= 0
  Non-trivial #2 (ray 15): mu(123)*mu(12|3) + mu(123)*mu(13|2) + mu(123)*mu(1|23) - mu(123)*mu(1|2|3) + 2*mu(12|3)*mu(13|2) + mu(12|3)*mu(1|23) + mu(13|2)*mu(1|23) + mu(1|23)*mu(1|2|3) >= 0
  Non-trivial #3 (ray 16): mu(1|23)^2 + mu(123)*mu(12|3) + mu(123)*mu(13|2) - mu(123)*mu(1|2|3) + 2*mu(12|3)*mu(13|2) + mu(12|3)*mu(1|23) + mu(13|2)*mu(1|23) + mu(1|23)*mu(1|2|3) >= 0

Summary: 14 trivial positivity + 3 non-trivial = 17 total


## Section 4: Identifying the Non-Trivial Rays

The 3 non-trivial extremal rays have a precise relationship to the paper's named inequalities:

- **Ray 1** = Equation (12) = Aas inequality: $\mu(123)\mu(1|2|3) - \mu(12|3)\mu(13|2) - \mu(12|3)\mu(1|23) - \mu(13|2)\mu(1|23) \ge 0$
- **Ray 16** = Equation (11): the extremal **strengthening** of Inequality (7)
- **Ray 15**: a third extremal inequality without a named counterpart in the paper

Inequality (7) (Proposition 10.1) states
$\mu(1|2\cap 1|3)\,\mu(12\cup 13) \le \mu(12|3) + \mu(13|2) + \mu(1|23)$.
When homogenized using $\sum_p \mu(p) = 1$, it becomes a quadratic form in the $\Phi$-coordinates
that differs from Ray 16 (Equation 11) by exactly $\mu(12|3)^2 + \mu(13|2)^2 \ge 0$.
Thus **Ray 16 implies Inequality (7)**, and is strictly stronger.

## Section 5: Computational Summary

### Pipeline dimensions

| Stage | $m=2$ | $m=4$ |
|-------|-------|-------|
| Feasible tuples $|F|$ | 139 | 1265 |
| $\varphi$-variables ($m \cdot B_3^2$) | 50 | 100 |
| Symmetric coordinates ($\binom{B_3+1}{2}$) | 15 | 15 |
| Feasibility constraint rows | 139 | 1265 |
| Equality encoding rows ($2 \times 15$) | 30 | 30 |
| Extended matrix size | $169 \times 65$ | $1295 \times 115$ |
| Projected dimension | 15 | 15 |

### Performance

In [10]:
println("Performance summary (HiGHS solver):\n")
println("  m  |  |F|  | rays | facets | time (s) | iters | LP calls")
println("  ---|-------|------|--------|----------|-------|--------")
tuple_counts = [25, 139, 570, 1265]
for m in 1:4
    result = enumerate_quiet(3, m)
    pr = result.projection_result
    @printf("  %d  | %5d | %4d | %6d | %8.2f | %5d | %6d\n",
        m, tuple_counts[m], length(result.rays), length(result.facet_normals),
        result.timing, pr.iterations, pr.lp_calls)
end

Performance summary (HiGHS solver):

  m  |  |F|  | rays | facets | time (s) | iters | LP calls
  ---|-------|------|--------|----------|-------|--------


  1  |    25 |   15 |     15 |     0.05 |     1 |     15


  2  |   139 |   15 |     15 |     0.46 |     1 |     15


  3  |   570 |   16 |     16 |     1.41 |     1 |     16


  4  |  1265 |   17 |     19 |     4.15 |     2 |     21


### Notes on $n=4$

For $n=4$ vertices, $B_4 = 15$ partitions yield $\binom{15+1}{2} = 120$ symmetric coordinates.
The extended matrix dimensions grow correspondingly:
- $m \cdot B_4^2 = 225m$ feasible-potential variables
- 240 equality-encoding rows ($2 \times 120$)
- Projected dimension: 120

Preliminary runs with $n=4$, $m=1$ (225 tuples) converge in ~6s with HiGHS,
yielding 120 rays (the simplicial positivity cone, as expected for a single tree).
Full $n=4$ enumeration with $m > 1$ requires the D1 oracle to generate feasible tuples
beyond the archived $n=3$ paper family.